# Mastodon — Spark Structured Streaming

This notebook consumes Mastodon posts from Kafka, cleans and filters
the data, stores posts in PostgreSQL, and computes streaming statistics.

Architecture: Mastodon API → Python producer → Kafka → Spark → PostgreSQL.

The producer polls the #AI timeline every 10 seconds.
Publication timestamps and analytical windows use UTC.
Run sections 1–8 in order. Run section 9 only to stop the demonstration.

The preview runs asynchronously: wait for records before stopping it
in the next cell.

Do not run the standalone Spark streaming script and this notebook
at the same time: they share the PostgreSQL checkpoint.

Keep checkpoint folders and Kafka/PostgreSQL volumes when restarting.

In [2]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql import types as T

spark = (
    SparkSession.builder
    .appName("MastodonStreamingNotebook")
    .master("local[2]")
    .config("spark.sql.session.timeZone", "UTC")
    .config("spark.sql.shuffle.partitions", "4")
    .config("spark.jars.ivy", "/tmp/spark-ivy")
    .config(
        "spark.jars.packages",
        "org.apache.spark:spark-sql-kafka-0-10_2.12:3.5.3,"
        "org.postgresql:postgresql:42.7.4",
    )
    .getOrCreate()
)

spark.sparkContext.setLogLevel("ERROR")


In [3]:
print("Spark version:", spark.version)
spark.range(5).show()

Spark version: 3.5.3
+---+
| id|
+---+
|  0|
|  1|
|  2|
|  3|
|  4|
+---+



## 1. Kafka ingestion

Spark reads JSON messages from the `mastodon_stream` topic.
An explicit schema defines the expected fields.

On the first run, `earliest` reads the available Kafka history.
On subsequent runs, the checkpoint determines where processing resumes.

In [6]:
schema = T.StructType([
    T.StructField("toot_id", T.StringType()),
    T.StructField("created_at", T.StringType()),
    T.StructField("user_id", T.StringType()),
    T.StructField("username", T.StringType()),
    T.StructField("content", T.StringType()),
    T.StructField("language", T.StringType()),
    T.StructField("hashtags", T.ArrayType(T.StringType())),
    T.StructField("favourites_count", T.IntegerType()),
    T.StructField("reblogs_count", T.IntegerType()),
])

raw_stream = (
    spark.readStream
    .format("kafka")
    .option("kafka.bootstrap.servers", "kafka:29092")
    .option("subscribe", "mastodon_stream")
    .option("startingOffsets", "earliest")
    .option("maxOffsetsPerTrigger", "1000")
    .load()
)

parsed_stream = (
    raw_stream
    .select(
        F.from_json(
            F.col("value").cast("string"),
            schema,
        ).alias("post")
    )
    .select("post.*")
)

print("Streaming DataFrame:", parsed_stream.isStreaming)
parsed_stream.printSchema()

Streaming DataFrame: True
root
 |-- toot_id: string (nullable = true)
 |-- created_at: string (nullable = true)
 |-- user_id: string (nullable = true)
 |-- username: string (nullable = true)
 |-- content: string (nullable = true)
 |-- language: string (nullable = true)
 |-- hashtags: array (nullable = true)
 |    |-- element: string (containsNull = true)
 |-- favourites_count: integer (nullable = true)
 |-- reblogs_count: integer (nullable = true)



## 2. Cleaning and language filtering

HTML is converted to plain text, and publication dates are parsed
as timestamps. Records with missing identifiers, invalid dates or
empty content are excluded.

All valid posts remain available for storage.
English posts are selected separately for streaming statistics,
providing a consistent language for the analysis.

In [8]:
from html.parser import HTMLParser


class TextExtractor(HTMLParser):
    def __init__(self):
        super().__init__(convert_charrefs=True)
        self.parts = []

    def handle_data(self, data):
        self.parts.append(data)

    def handle_starttag(self, tag, attrs):
        if tag in ("p", "br", "div"):
            self.parts.append(" ")

    def handle_endtag(self, tag):
        if tag in ("p", "div"):
            self.parts.append(" ")


@F.udf(returnType=T.StringType())
def clean_html(content):
    if content is None:
        return None

    parser = TextExtractor()
    parser.feed(content)
    parser.close()

    return " ".join("".join(parser.parts).split())


cleaned_stream = (
    parsed_stream
    .withColumn("created_at", F.to_timestamp("created_at"))
    .withColumn("content", clean_html(F.col("content")))
    .withColumn("language", F.lower(F.trim(F.col("language"))))
    .filter(
        F.col("toot_id").isNotNull()
        & (F.length(F.trim(F.col("toot_id"))) > 0)
        & F.col("user_id").isNotNull()
        & F.col("created_at").isNotNull()
        & (F.length("content") > 0)
    )
    .withColumn("text_length", F.length("content"))
)

english_stream = cleaned_stream.filter(
    F.col("language") == "en"
)

print("Cleaned stream ready.")
print("English stream ready for statistics.")

Cleaned stream ready.
English stream ready for statistics.


## 3. Preview

A temporary console query displays cleaned English posts.
It is stopped before launching the PostgreSQL queries.

In [12]:
from pathlib import Path

PROJECT_ROOT = Path("/opt/spark/work-dir")
CHECKPOINT_ROOT = PROJECT_ROOT / "checkpoints"

# Éviter de lancer deux aperçus en réexécutant cette cellule.
if "preview_query" in globals() and preview_query.isActive:
    preview_query.stop()

preview_query = (
    english_stream
    .select(
        "toot_id",
        "created_at",
        "username",
        "language",
        F.substring("content", 1, 120).alias("content_preview"),
        "text_length",
    )
    .writeStream
    .queryName("english_preview")
    .format("console")
    .outputMode("append")
    .option("truncate", "false")
    .option("numRows", "5")
    .option(
        "checkpointLocation",
        str(CHECKPOINT_ROOT / "notebook_preview"),
    )
    .trigger(processingTime="10 seconds")
    .start()
)

print("Preview started:", preview_query.isActive)

Preview started: True


26/10/08 08:05:52 WARN ResolveWriteToStream: spark.sql.adaptive.enabled is not supported in streaming DataFrames/Datasets and will be disabled.
26/10/08 08:05:53 WARN AdminClientConfig: These configurations '[key.deserializer, value.deserializer, enable.auto.commit, max.poll.records, auto.offset.reset]' were supplied but are not used yet.
26/10/08 08:06:10 WARN KafkaDataConsumer: KafkaDataConsumer is not running in UninterruptibleThread. It may hang when KafkaDataConsumer's methods are interrupted because of KAFKA-1894


-------------------------------------------
Batch: 2
-------------------------------------------
+------------------+-------------------+-----------------------+--------+------------------------------------------------------------------------------------------------------------------------+-----------+
|toot_id           |created_at         |username               |language|content_preview                                                                                                         |text_length|
+------------------+-------------------+-----------------------+--------+------------------------------------------------------------------------------------------------------------------------+-----------+
|117404254924861589|2026-10-08 08:05:59|Econopass@flipboard.com|en      |In the shadow of a rumored AI proof of one of the biggest problems in their field, three computer scientists rushed to p|460        |
+------------------+-------------------+-----------------------+--------+--

26/10/08 08:08:30 WARN KafkaDataConsumer: KafkaDataConsumer is not running in UninterruptibleThread. It may hang when KafkaDataConsumer's methods are interrupted because of KAFKA-1894


-------------------------------------------
Batch: 3
-------------------------------------------
+------------------+-----------------------+--------+--------+------------------------------------------------------------------------------------------------------------------------+-----------+
|toot_id           |created_at             |username|language|content_preview                                                                                                         |text_length|
+------------------+-----------------------+--------+--------+------------------------------------------------------------------------------------------------------------------------+-----------+
|117404264239213859|2026-10-08 08:08:23.066|stabby  |en      |Curious, is #ai useful yet? I remember in 2022 it was hallucinating like mad, has that bug been fixed yet, or is it stil|132        |
+------------------+-----------------------+--------+--------+---------------------------------------------------------

In [13]:
preview_query.stop()
print("Preview stopped.")

Preview stopped.


## 4. Store cleaned posts in PostgreSQL

Each micro-batch is written in a PostgreSQL transaction.
The unique `toot_id` constraint prevents duplicate posts.

All valid languages are stored for historical analysis.
The English-only stream will be used for streaming statistics.

If a batch fails, the error is propagated to Spark.
A replay can safely retry the inserts using ON CONFLICT DO NOTHING.

In [14]:
import psycopg2


def write_posts(batch_df, batch_id):
    connection = psycopg2.connect(connect_timeout=10)

    processed = 0
    inserted = 0

    sql = """
        INSERT INTO toots (
            toot_id, created_at, user_id, username,
            content, language, hashtags,
            favourites_count, reblogs_count
        )
        VALUES (%s, %s, %s, %s, %s, %s, %s, %s, %s)
        ON CONFLICT (toot_id) DO NOTHING
    """

    try:
        with connection:
            with connection.cursor() as cursor:
                rows = (
                    batch_df
                    .select(
                        "toot_id",
                        F.date_format(
                            "created_at",
                            "yyyy-MM-dd HH:mm:ss.SSSSSS",
                        ).alias("created_at"),
                        "user_id",
                        "username",
                        "content",
                        "language",
                        "hashtags",
                        "favourites_count",
                        "reblogs_count",
                    )
                    .toLocalIterator()
                )

                for row in rows:
                    cursor.execute(
                        sql,
                        (
                            row.toot_id,
                            row.created_at,
                            row.user_id,
                            row.username,
                            row.content,
                            row.language,
                            row.hashtags or [],
                            row.favourites_count or 0,
                            row.reblogs_count or 0,
                        ),
                    )

                    processed += 1
                    inserted += cursor.rowcount

        print(
            f"Batch {batch_id}: "
            f"{inserted} inserted, "
            f"{processed - inserted} duplicates ignored.",
            flush=True,
        )

    finally:
        connection.close()

In [15]:
if "posts_query" in globals() and posts_query.isActive:
    posts_query.stop()

posts_query = (
    cleaned_stream.writeStream
    .queryName("postgres_posts")
    .foreachBatch(write_posts)
    .outputMode("append")
    .option(
        "checkpointLocation",
        str(CHECKPOINT_ROOT / "postgres"),
    )
    .trigger(processingTime="10 seconds")
    .start()
)

print("PostgreSQL writer active:", posts_query.isActive)

PostgreSQL writer active: True


26/10/08 08:15:14 WARN ResolveWriteToStream: spark.sql.adaptive.enabled is not supported in streaming DataFrames/Datasets and will be disabled.
26/10/08 08:15:17 WARN AdminClientConfig: These configurations '[key.deserializer, value.deserializer, enable.auto.commit, max.poll.records, auto.offset.reset]' were supplied but are not used yet.
26/10/08 08:15:18 WARN KafkaDataConsumer: KafkaDataConsumer is not running in UninterruptibleThread. It may hang when KafkaDataConsumer's methods are interrupted because of KAFKA-1894
26/10/08 08:15:18 WARN KafkaDataConsumer: KafkaDataConsumer is not running in UninterruptibleThread. It may hang when KafkaDataConsumer's methods are interrupted because of KAFKA-1894
26/10/08 08:15:18 WARN KafkaDataConsumer: KafkaDataConsumer is not running in UninterruptibleThread. It may hang when KafkaDataConsumer's methods are interrupted because of KAFKA-1894
26/10/08 08:15:18 WARN KafkaDataConsumer: KafkaDataConsumer is not running in UninterruptibleThread. It may

Batch 42: 75 inserted, 0 duplicates ignored.


26/10/08 08:15:20 WARN KafkaDataConsumer: KafkaDataConsumer is not running in UninterruptibleThread. It may hang when KafkaDataConsumer's methods are interrupted because of KAFKA-1894
26/10/08 08:15:20 WARN KafkaDataConsumer: KafkaDataConsumer is not running in UninterruptibleThread. It may hang when KafkaDataConsumer's methods are interrupted because of KAFKA-1894
26/10/08 08:15:20 WARN KafkaDataConsumer: KafkaDataConsumer is not running in UninterruptibleThread. It may hang when KafkaDataConsumer's methods are interrupted because of KAFKA-1894
26/10/08 08:15:20 WARN KafkaDataConsumer: KafkaDataConsumer is not running in UninterruptibleThread. It may hang when KafkaDataConsumer's methods are interrupted because of KAFKA-1894
26/10/08 08:15:20 WARN KafkaDataConsumer: KafkaDataConsumer is not running in UninterruptibleThread. It may hang when KafkaDataConsumer's methods are interrupted because of KAFKA-1894
26/10/08 08:15:20 WARN KafkaDataConsumer: KafkaDataConsumer is not running in Un

Batch 43: 1 inserted, 0 duplicates ignored.
Batch 44: 1 inserted, 0 duplicates ignored.
Batch 45: 1 inserted, 0 duplicates ignored.
Batch 46: 2 inserted, 0 duplicates ignored.
Batch 47: 1 inserted, 0 duplicates ignored.
Batch 48: 1 inserted, 0 duplicates ignored.
Batch 49: 1 inserted, 0 duplicates ignored.
Batch 50: 1 inserted, 0 duplicates ignored.
Batch 51: 1 inserted, 0 duplicates ignored.
Batch 52: 1 inserted, 0 duplicates ignored.
Batch 53: 1 inserted, 0 duplicates ignored.
Batch 54: 1 inserted, 0 duplicates ignored.
Batch 55: 1 inserted, 0 duplicates ignored.
Batch 56: 1 inserted, 0 duplicates ignored.
Batch 57: 1 inserted, 0 duplicates ignored.
Batch 58: 1 inserted, 0 duplicates ignored.
Batch 59: 1 inserted, 0 duplicates ignored.
Batch 60: 1 inserted, 0 duplicates ignored.
Batch 61: 1 inserted, 0 duplicates ignored.
Batch 62: 1 inserted, 0 duplicates ignored.
Batch 63: 1 inserted, 0 duplicates ignored.
Batch 64: 1 inserted, 0 duplicates ignored.
Statistics batch 0: 446 windows 

## 5. Verify stored data

The following queries check the total number of stored posts,
display recent records and verify that toot identifiers are unique.
These checks can be rerun while the streaming writer is active.

In [19]:
import pandas as pd


def read_postgres(sql):
    connection = psycopg2.connect(connect_timeout=10)

    try:
        with connection.cursor() as cursor:
            cursor.execute(sql)

            columns = [
                description[0]
                for description in cursor.description
            ]

            return pd.DataFrame(
                cursor.fetchall(),
                columns=columns,
            )
    finally:
        connection.close()


display(read_postgres("""
    SELECT
        COUNT(*) AS total_posts,
        COUNT(DISTINCT toot_id) AS distinct_posts,
        MAX(created_at) AS latest_publication_utc
    FROM toots
"""))

display(read_postgres("""
    SELECT
        toot_id,
        created_at AS publication_utc,
        username,
        language,
        LEFT(content, 120) AS content_preview
    FROM toots
    ORDER BY created_at DESC
    LIMIT 5
"""))

,total_posts,distinct_posts,latest_publication_utc
0,1411,1411,2026-10-08 08:19:17


,toot_id,publication_utc,username,language,content_preview
0,117404307174154026,2026-10-08 08:19:17,gamingonlinux@mastodon.world,en,Unity were already moving forwards with Unity ...
1,117404299698405348,2026-10-08 08:17:22,oneoveralpha@chinwag.org,en,Brain for Rent and other stories contains five...
2,117404290033241853,2026-10-08 08:14:52,fasnix@littleone.littlefedi.social,None,"""Kelsey Piper argues that, once you have a dec..."
3,117404287971030634,2026-10-08 08:14:22,feed@igeek.gamer-geek-news.com,en,🎮 Fireshine Games acquires Necesse dev Fair Ga...
4,117404284992963592,2026-10-08 08:13:37,fediboard_science@flipboard.social,en,Researchers raced to beat an AI proof of the U...


## 6. English posts per hour

Posts are grouped into one-hour windows using their publication time
(event time), in UTC.

Distinct toot identifiers are counted to avoid counting replayed posts twice.
A one-day watermark bounds the retained state. Posts arriving more than
one day behind the latest observed event time may be excluded.

Counts are provisional while a window remains open.

In [39]:
connection = psycopg2.connect(connect_timeout=10)

try:
    with connection:
        with connection.cursor() as cursor:
            cursor.execute("""
                CREATE TABLE IF NOT EXISTS streaming_posts_per_hour (
                    window_start TIMESTAMP PRIMARY KEY,
                    window_end TIMESTAMP NOT NULL,
                    post_count BIGINT NOT NULL,
                    updated_at TIMESTAMPTZ NOT NULL DEFAULT NOW()
                )
            """)
finally:
    connection.close()

print("Hourly statistics table ready.")

Hourly statistics table ready.


In [45]:
hour_stats = (
    english_stream
    .withWatermark("created_at", "1 day")
    .groupBy(
        F.window("created_at", "1 hour")
    )
    .agg(
        F.size(
            F.collect_set("toot_id")
        ).cast("long").alias("post_count")
    )
    .select(
        F.col("window.start").alias("window_start"),
        F.col("window.end").alias("window_end"),
        "post_count",
    )
)

hour_stats.printSchema()

root
 |-- window_start: timestamp (nullable = true)
 |-- window_end: timestamp (nullable = true)
 |-- post_count: long (nullable = false)



In [46]:
def write_hour_stats(batch_df, batch_id):
    connection = psycopg2.connect(connect_timeout=10)
    updated = 0

    sql = """
        INSERT INTO streaming_posts_per_hour (
            window_start, window_end, post_count
        )
        VALUES (%s, %s, %s)

        ON CONFLICT (window_start)
        DO UPDATE SET
            window_end = EXCLUDED.window_end,
            post_count = EXCLUDED.post_count,
            updated_at = NOW()
    """

    try:
        with connection:
            with connection.cursor() as cursor:
                rows = (
                    batch_df
                    .select(
                        F.date_format(
                            "window_start",
                            "yyyy-MM-dd HH:mm:ss",
                        ).alias("window_start"),
                        F.date_format(
                            "window_end",
                            "yyyy-MM-dd HH:mm:ss",
                        ).alias("window_end"),
                        "post_count",
                    )
                    .toLocalIterator()
                )

                for row in rows:
                    cursor.execute(
                        sql,
                        (
                            row.window_start,
                            row.window_end,
                            row.post_count,
                        ),
                    )
                    updated += 1

        print(
            f"Hourly statistics batch {batch_id}: "
            f"{updated} windows updated.",
            flush=True,
        )

    finally:
        connection.close()

In [47]:
# Éviter de lancer deux requêtes horaires.
if "hour_query" in globals() and hour_query.isActive:
    hour_query.stop()

hour_query = (
    hour_stats.writeStream
    .queryName("english_posts_per_hour")
    .foreachBatch(write_hour_stats)
    .outputMode("update")
    .option(
        "checkpointLocation",
        str(CHECKPOINT_ROOT / "english_posts_per_hour"),
    )
    .trigger(processingTime="10 seconds")
    .start()
)

print("Hourly statistics active:", hour_query.isActive)

Hourly statistics active: True


In [48]:
spark.sparkContext.setLogLevel("ERROR")

print("Active:", hour_query.isActive)
print("Error:", hour_query.exception())

display(read_postgres("""
    SELECT
        window_start AS hour_utc,
        window_end,
        post_count
    FROM streaming_posts_per_hour
    ORDER BY window_start DESC
    LIMIT 10
"""))

Active: True
Error: None


,hour_utc,window_end,post_count
0,2026-10-08 09:00:00,2026-10-08 10:00:00,30
1,2026-10-08 08:00:00,2026-10-08 09:00:00,62
2,2026-10-08 07:00:00,2026-10-08 08:00:00,48
3,2026-10-08 06:00:00,2026-10-08 07:00:00,38
4,2026-10-08 05:00:00,2026-10-08 06:00:00,59
5,2026-10-08 04:00:00,2026-10-08 05:00:00,45
6,2026-10-08 03:00:00,2026-10-08 04:00:00,51
7,2026-10-08 02:00:00,2026-10-08 03:00:00,39
8,2026-10-08 01:00:00,2026-10-08 02:00:00,40
9,2026-10-08 00:00:00,2026-10-08 01:00:00,52


## 7. Average post length per user and hour

English posts are grouped by user and one-hour publication-time window,
using UTC timestamps.

The average length is measured in characters after HTML cleaning.
Repeated identical posts are counted once.

A one-day watermark limits the retained state.
Results are updated in PostgreSQL every micro-batch and remain
provisional while the window is open.

In [49]:
connection = psycopg2.connect(connect_timeout=10)

try:
    with connection:
        with connection.cursor() as cursor:
            cursor.execute("""
                CREATE TABLE IF NOT EXISTS streaming_user_stats_hourly (
                    window_start TIMESTAMP NOT NULL,
                    window_end TIMESTAMP NOT NULL,
                    user_id VARCHAR(100) NOT NULL,
                    post_count BIGINT NOT NULL,
                    avg_text_length DOUBLE PRECISION NOT NULL,
                    updated_at TIMESTAMPTZ NOT NULL DEFAULT NOW(),
                    PRIMARY KEY (window_start, user_id)
                )
            """)
finally:
    connection.close()

print("Hourly user statistics table ready.")

Hourly user statistics table ready.


In [50]:
user_stats_hourly = (
    english_stream
    .withWatermark("created_at", "1 day")
    .groupBy(
        F.window("created_at", "1 hour"),
        "user_id",
    )
    .agg(
        F.collect_set(
            F.struct("toot_id", "text_length")
        ).alias("posts")
    )
    .select(
        F.col("window.start").alias("window_start"),
        F.col("window.end").alias("window_end"),
        "user_id",
        F.size("posts").cast("long").alias("post_count"),
        F.expr("""
            aggregate(
                posts,
                CAST(0 AS BIGINT),
                (total, post) -> total + post.text_length
            ) / size(posts)
        """).cast("double").alias("avg_text_length"),
    )
)

user_stats_hourly.printSchema()

root
 |-- window_start: timestamp (nullable = true)
 |-- window_end: timestamp (nullable = true)
 |-- user_id: string (nullable = true)
 |-- post_count: long (nullable = false)
 |-- avg_text_length: double (nullable = true)



In [51]:
def write_user_stats_hourly(batch_df, batch_id):
    connection = psycopg2.connect(connect_timeout=10)
    updated = 0

    sql = """
        INSERT INTO streaming_user_stats_hourly (
            window_start,
            window_end,
            user_id,
            post_count,
            avg_text_length
        )
        VALUES (%s, %s, %s, %s, %s)

        ON CONFLICT (window_start, user_id)
        DO UPDATE SET
            window_end = EXCLUDED.window_end,
            post_count = EXCLUDED.post_count,
            avg_text_length = EXCLUDED.avg_text_length,
            updated_at = NOW()
    """

    try:
        with connection:
            with connection.cursor() as cursor:
                rows = (
                    batch_df
                    .select(
                        F.date_format(
                            "window_start",
                            "yyyy-MM-dd HH:mm:ss",
                        ).alias("window_start"),
                        F.date_format(
                            "window_end",
                            "yyyy-MM-dd HH:mm:ss",
                        ).alias("window_end"),
                        "user_id",
                        "post_count",
                        "avg_text_length",
                    )
                    .toLocalIterator()
                )

                for row in rows:
                    cursor.execute(
                        sql,
                        (
                            row.window_start,
                            row.window_end,
                            row.user_id,
                            row.post_count,
                            row.avg_text_length,
                        ),
                    )
                    updated += 1

        print(
            f"Hourly user statistics batch {batch_id}: "
            f"{updated} rows updated.",
            flush=True,
        )

    finally:
        connection.close()

In [53]:
# Éviter de lancer deux requêtes horaires.
if "user_query_hourly" in globals() and user_query_hourly.isActive:
    user_query_hourly.stop()

user_query_hourly = (
    user_stats_hourly.writeStream
    .queryName("english_user_statistics_hourly")
    .foreachBatch(write_user_stats_hourly)
    .outputMode("update")
    .option(
        "checkpointLocation",
        str(CHECKPOINT_ROOT / "english_user_statistics_hourly"),
    )
    .trigger(processingTime="10 seconds")
    .start()
)

In [54]:
print("Active:", user_query_hourly.isActive)
print("Error:", user_query_hourly.exception())

display(read_postgres("""
    SELECT
        window_start AS hour_utc,
        user_id,
        post_count,
        ROUND(avg_text_length::numeric, 1) AS avg_characters
    FROM streaming_user_stats_hourly
    ORDER BY window_start DESC, post_count DESC, user_id
    LIMIT 10
"""))

Active: True
Error: None


,hour_utc,user_id,post_count,avg_characters
0,2026-10-08 09:00:00,116070447239053927,8,454.3
1,2026-10-08 09:00:00,115061140460610278,4,7432.0
2,2026-10-08 09:00:00,116450756112437762,4,387.3
3,2026-10-08 09:00:00,116988275448638305,3,332.3
4,2026-10-08 09:00:00,112026408859289825,2,335.0
5,2026-10-08 09:00:00,113373281572068351,2,304.5
6,2026-10-08 09:00:00,116467552435501973,2,467.5
7,2026-10-08 09:00:00,107320075805792927,1,220.0
8,2026-10-08 09:00:00,109212695704203107,1,360.0
9,2026-10-08 09:00:00,109318490906315885,1,635.0


## 8. Streaming monitoring

Check whether each streaming query is active and inspect any errors.
Each query processes new Kafka data every 10 seconds.

In [55]:
queries = {
    "Posts → PostgreSQL": posts_query,
    "Posts per hour": hour_query,
    "User statistics per hour": user_query_hourly,
}

for name, query in queries.items():
    print(f"\n{name}")
    print("Active:", query.isActive)
    print("Error:", query.exception())
    print("Status:", query.status["message"])


Posts → PostgreSQL
Active: True
Error: None
Status: Waiting for next trigger

Posts per hour
Active: True
Error: None
Status: Waiting for next trigger

User statistics per hour
Active: True
Error: None
Status: Waiting for next trigger


## 9. Stop streaming queries

Run the following cell only when the demonstration is finished.
Checkpoints are retained so processing can resume on restart.

In [ ]:
for query in list(spark.streams.active):
    query.stop()
    print(f"Stopped: {query.name}")

print("Active queries remaining:", len(spark.streams.active))

To resume in the same kernel, rerun the start cells for posts_query,
hour_query and user_query_hourly.

After restarting the kernel, run the notebook again through section 8.